# 05 — Vector Inspection & Diachronic Concept Trajectories

### What this notebook does:
1. **L2-Normalizes Word2Vec Vectors** (`.kv`) for unit cosine geometry without altering raw model weights.
2. **Projects Target Words onto Custom Semantic Axes** (e.g. *fair* vs *biased*, *supportive* vs *toxic*) over time.
3. **Calculates Cross-Seed Neighbor Stability (Jaccard)** across random seeds to ensure trends are robust semantic shifts rather than random sampling noise.
4. **Generates Diagnostic Plots & Markdown Reports** summarizing semantic drift and vocabulary coverage.
5. **Creates a Project-Wide `RUN_SUMMARY.md`** tracking models, vector counts, and trajectory datasets.

---

In [ ]:
# =============================================================================
# Cell 1 — USER SETTINGS & CONCEPT DEFINITIONS (EDIT THIS CELL)
# =============================================================================

# 1. DEFINE CONCEPTS & SEMANTIC AXES TO TRACK OVER TIME:
#    - name: short descriptive identifier
#    - targets: words whose movement across time you want to inspect
#    - pole_a: positive semantic anchor words
#    - pole_b: opposing semantic anchor words
#    - anchors: optional reference nouns to measure baseline cosine distance
CONCEPTS = [
    {
        "name": "moderator_sentiment",
        "targets": ["moderator", "mod", "admin"],
        "pole_a": ["fair", "helpful", "transparent", "reasonable", "kind"],
        "pole_b": ["biased", "corrupt", "abusive", "toxic", "unfair"],
        "anchors": ["community", "rules", "post"]
    }
]

# 2. TARGET FILTERING (Optional):
#    - Set to None to analyze all subreddits/periods, or specify filters:
MODEL_FILTER_SUB = None   # e.g. 'AskAcademia' or 'ALL_REDDIT'
PERIOD_FILTER = None      # e.g. ['w2v__askacademia__2019q1']
SEED_FILTER = None        # e.g. [1047] or None to use all available seeds

# 3. ANALYSIS CONTROLS:
DO_NORMALIZE = True       # Generate .kv KeyedVectors with fill_norms() if missing
K_NEIGHBORS = 20          # Number of nearest neighbors for Jaccard overlap

print(f"Vector Inspection Configuration:")
print(f"  • Defined Concepts: {[c['name'] for c in CONCEPTS]}")
print(f"  • Target Subreddit: {MODEL_FILTER_SUB or 'ALL'}")
print(f"  • Nearest Neighbors (K): {K_NEIGHBORS}")

In [ ]:
# Bootstrap: locate the repo root and add it to sys.path so the shared 'src'
# package is importable regardless of the notebook's current working directory.
import os, sys
from pathlib import Path
def _has_root_marker(p):
    try:
        return p.is_dir() and (p / "config/project_config.yaml").is_file()
    except OSError:
        return False
def _find_root(start):
    for p in [start, *start.parents]:
        if _has_root_marker(p):
            return p
    for depth in (1, 2):
        for sub in start.glob("/".join(["*"] * depth)):
            if sub.is_dir() and _has_root_marker(sub):
                return sub
    return None
_repo_root = _find_root(Path.cwd().resolve())
if _repo_root is not None and str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))
# Cell 2 — Setup: Imports, Config, Paths, and Logging
import os, sys, csv, json, gc, hashlib, datetime
from pathlib import Path
from collections import defaultdict
import yaml
import numpy as np

from src.paths import get_project_root
from src.storage import atomic_write_text, sha256_file, save_gensim_atomic

ROOT = get_project_root()
cfg_path = ROOT / "config/project_config.yaml"
cfg = yaml.safe_load(open(cfg_path, encoding="utf-8"))
CLAIM_FLOOR = cfg["embeddings"].get("interpretation_floor", 100)

try:
    from gensim.models import Word2Vec, KeyedVectors
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "gensim"])
    from gensim.models import Word2Vec, KeyedVectors

import logging
ts = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
LOGP = ROOT / f"logs/05_vectors_inspect__{ts}__cfg-{cfg['config_version']}.log"
LOGP.parent.mkdir(parents=True, exist_ok=True)
lg = logging.getLogger("v5"); lg.setLevel(logging.INFO); lg.handlers.clear()
fh = logging.FileHandler(LOGP); fh.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
sh = logging.StreamHandler(sys.stdout); sh.setLevel(logging.WARNING)
lg.addHandler(fh); lg.addHandler(sh)

for d in ["vectors", "diagnostics/semantic_axes"]:
    (ROOT / d).mkdir(parents=True, exist_ok=True)

OUTD = ROOT / "diagnostics/semantic_axes"
today = datetime.datetime.now(datetime.timezone.utc).date().isoformat()

print(f"Setup complete | Config version: {cfg['config_version']} | Min word claim floor: {CLAIM_FLOOR}")

In [ ]:
# Cell 3 — L2-Normalize and Export KeyedVectors (.kv)
# Creates lightweight .kv files with precomputed unit vectors for fast cosine math.

TMAN = ROOT / "manifests/training_manifest.csv"
rows = list(csv.DictReader(open(TMAN, encoding="utf-8"))) if TMAN.exists() else []
T_COLS = list(rows[0].keys()) if rows else []
n_norm = n_skip = n_fail = 0

if DO_NORMALIZE:
    for r in rows:
        if r["status"] != "complete": continue
        if MODEL_FILTER_SUB and r["subreddit_or_group"] != MODEL_FILTER_SUB: continue
        if PERIOD_FILTER and r["model_id"] not in PERIOD_FILTER: continue
        
        mp = Path(r["model_path"])
        vname = mp.stem.replace("w2v__", "vectors_norm__") + ".kv"
        vp = ROOT / "vectors" / vname
        
        if r.get("vectors_path") and Path(r["vectors_path"]).exists() and r.get("vectors_sha256"):
            if sha256_file(Path(r["vectors_path"])) == r["vectors_sha256"]:
                n_skip += 1
                continue
        
        try:
            m = Word2Vec.load(str(mp))
            m.wv.fill_norms()
            save_gensim_atomic(m.wv, vp)
            assert vp.stat().st_size > 0
            r["vectors_path"] = str(vp)
            r["vectors_sha256"] = sha256_file(vp)
            del m; gc.collect()
            n_norm += 1
            if n_norm % 5 == 0: print(f"  Normalized {n_norm} models...")
        except Exception as e:
            lg.error(f"Normalization failed for {r['model_id']}: {e}")
            n_fail += 1

    if T_COLS and n_norm > 0:
        tmp = TMAN.with_suffix(".tmp")
        f = open(tmp, "w", newline="", encoding="utf-8")
        w = csv.DictWriter(f, fieldnames=T_COLS)
        w.writeheader(); w.writerows(rows); f.flush(); os.fsync(f.fileno()); f.close()
        os.replace(tmp, TMAN)

print(f"Normalization step complete: {n_norm} newly created, {n_skip} already verified, {n_fail} failed.")

In [ ]:
# Cell 4 — Index Models Grouped by Subreddit and Period
groups = defaultdict(list)
for r in rows:
    if r["status"] != "complete": continue
    if MODEL_FILTER_SUB and r["subreddit_or_group"] != MODEL_FILTER_SUB: continue
    if SEED_FILTER and int(r["seed"]) not in SEED_FILTER: continue
    
    vp = Path(r.get("vectors_path", "")) if r.get("vectors_path") else None
    mp = Path(r["model_path"])
    target_p = vp if (vp and vp.exists() and vp.stat().st_size > 0) else mp
    if target_p.exists() and target_p.stat().st_size > 0:
        groups[(r["subreddit_or_group"], r["period_id"])].append((int(r["seed"]), target_p, target_p == vp))

for k in groups: groups[k].sort()
order = sorted(groups)
print(f"Indexed {len(groups)} distinct time periods across {sum(len(v) for v in groups.values())} seed models.")
if not groups:
    print("WARNING: No complete models found in manifest. Please run Notebook 03_04 first.")

In [ ]:
# Cell 5 — Metric Engine: Semantic Projection, Anchor Distance & Jaccard Stability
def cos(a, b):
    d = float(np.linalg.norm(a) * np.linalg.norm(b))
    return float(np.dot(a, b) / d) if d else 0.0

def mean_pairwise(vecs):
    if len(vecs) < 2: return None
    s = n = 0.0
    for i in range(len(vecs)):
        for j in range(i + 1, len(vecs)):
            s += cos(vecs[i], vecs[j]); n += 1
    return s / n

def jaccard(a, b):
    a, b = set(a), set(b)
    return len(a & b) / len(a | b) if (a | b) else 0.0

def analyze_model(wv, concept, K):
    wv.fill_norms()
    out = {"coverage": {}, "coherence": {}, "freq": {}}
    present = {}
    for pole in ("pole_a", "pole_b"):
        ok, miss = [], []
        for w in concept.get(pole, []) or []:
            if w in wv:
                c = int(wv.get_vecattr(w, "count")) if hasattr(wv, "get_vecattr") else 1000
                out["freq"][w] = c
                (ok if c >= CLAIM_FLOOR else miss).append(w)
            else:
                miss.append(w)
        present[pole] = ok
        out["coverage"][pole] = {"kept": ok, "missing_or_rare": miss}
        
    for w in (concept.get("targets", []) + concept.get("anchors", [])):
        out["freq"][w] = int(wv.get_vecattr(w, "count")) if (hasattr(wv, "get_vecattr") and w in wv) else (1000 if w in wv else 0)
        
    va = [wv.get_vector(w, norm=True) for w in present["pole_a"]]
    vb = [wv.get_vector(w, norm=True) for w in present["pole_b"]]
    out["coherence"]["pole_a"] = mean_pairwise(va)
    out["coherence"]["pole_b"] = mean_pairwise(vb)
    out["separation"] = float(np.mean([cos(a, b) for a in va for b in vb])) if va and vb else None
    
    axis = None
    if va and vb:
        axis = np.mean(va, axis=0) - np.mean(vb, axis=0)
        norm = np.linalg.norm(axis)
        axis = axis / norm if norm else None
        
    out["proj"] = {}
    for w in concept.get("targets", []):
        if w in wv and axis is not None:
            out["proj"][w] = float(np.dot(wv.get_vector(w, norm=True), axis))
        else:
            out["proj"][w] = None
            
    out["anchors"] = {}
    for a in concept.get("anchors", []):
        if a in wv:
            out["anchors"][a] = {w: cos(wv.get_vector(a, norm=True), wv.get_vector(w, norm=True)) for w in concept.get("targets", []) if w in wv}
            
    out["neighbors"] = {}
    for w in concept.get("targets", []):
        if w in wv:
            out["neighbors"][w] = [x for x, _ in wv.most_similar(w, topn=K)]
        else:
            out["neighbors"][w] = []
    return out

trows_out, cov_summary = [], {}
for c in CONCEPTS:
    cn = c["name"]
    for (grp, pid), seed_list in groups.items():
        by_seed = {}
        for (s, path, is_kv) in seed_list:
            try:
                wv = KeyedVectors.load(str(path)) if is_kv else Word2Vec.load(str(path)).wv
            except Exception as le:
                lg.warning(f"Failed to load {path}: {le}")
                continue
            an = analyze_model(wv, c, K_NEIGHBORS)
            by_seed[s] = an
            cov_summary[(cn, grp, pid, s)] = an["coverage"]
            
            for w, sc in an["proj"].items():
                trows_out.append([cn, grp, pid, s, "projection", w, sc, ""])
            for a, dt in an["anchors"].items():
                for w, sc in dt.items():
                    trows_out.append([cn, grp, pid, s, f"cos_anchor_{a}", w, sc, ""])
            for w, nb in an["neighbors"].items():
                trows_out.append([cn, grp, pid, s, "neighbors", w, "", ";".join(nb)])
            for w, cnt in an["freq"].items():
                trows_out.append([cn, grp, pid, s, "frequency", w, cnt, ""])
            for p, sc in an["coherence"].items():
                trows_out.append([cn, grp, pid, s, f"coherence_{p}", "", sc, ""])
            if an["separation"] is not None:
                trows_out.append([cn, grp, pid, s, "separation", "", an["separation"], ""])
            del wv; gc.collect()
            
        # Cross-seed Jaccard neighbor overlap within this period
        seeds_present = sorted(by_seed.keys())
        for w in c.get("targets", []):
            pair_j = []
            for i in range(len(seeds_present)):
                for j in range(i + 1, len(seeds_present)):
                    s1, s2 = seeds_present[i], seeds_present[j]
                    nb1 = by_seed[s1]["neighbors"].get(w, [])
                    nb2 = by_seed[s2]["neighbors"].get(w, [])
                    if nb1 and nb2: pair_j.append(jaccard(nb1, nb2))
            mean_j = float(np.mean(pair_j)) if pair_j else None
            trows_out.append([cn, grp, pid, "all_seeds", "cross_seed_neighbor_jaccard", w, mean_j, ""])

TRAJ = OUTD / "trajectories.csv"
tmp = TRAJ.with_suffix(".tmp")
f = open(tmp, "w", newline="", encoding="utf-8"); w = csv.writer(f)
w.writerow(["concept", "subreddit_or_group", "period_id", "seed", "metric", "word_or_pole", "value", "extra"])
w.writerows(trows_out); f.flush(); os.fsync(f.fileno()); f.close(); os.replace(tmp, TRAJ)
print(f"Extracted {len(trows_out)} trajectory data points -> Saved to {TRAJ}")

In [ ]:
# Cell 6 — Diagnostic Plots: Projection, Jaccard Overlap, and Frequency Guardrail
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def series(metric, cn, grp, word):
    d = defaultdict(dict)
    for r in trows_out:
        if r[0] == cn and r[1] == grp and r[4] == metric and r[5] == word:
            try: d[r[2]][r[3]] = float(r[6])
            except ValueError: pass
    return d

def neighbors_of(cn, grp, pid, seed, t):
    for r in trows_out:
        if (r[0], r[1], r[2], r[3], r[4], r[5]) == (cn, grp, pid, seed, "neighbors", t):
            return r[7].split(";") if r[7] else []
    return []

made_plots = []
for c in CONCEPTS:
    cn = c["name"]
    for grp in sorted({g for (g, p) in order}):
        pids = sorted({p for (g, p) in order if g == grp})
        if not pids: continue
        seeds = sorted({s for (g, p) in order if g == grp for (s, _, _) in groups[(g, p)]})
        
        for t in c.get("targets", []):
            fig, ax = plt.subplots(3, 1, figsize=(10, 11), sharex=True)
            
            # Panel 1: Neighbor Jaccard Stability vs Previous Period
            for s in seeds:
                js, prev = [], None
                for pid in pids:
                    nb = neighbors_of(cn, grp, pid, s, t)
                    js.append((len(set(nb) & set(prev)) / len(set(nb) | set(prev)) if prev is not None and nb and prev else None))
                    prev = nb
                ax[0].plot(pids, js, marker="o", label=f"seed {s}")
            ax[0].set_title(f"{cn} / '{t}' @ {grp} — Neighbor Stability (Jaccard K={K_NEIGHBORS})", fontweight="bold")
            ax[0].set_ylim(-0.05, 1.05); ax[0].legend(fontsize=8); ax[0].grid(True, alpha=0.3)
            
            # Panel 2: Semantic Axis Projection Over Time
            pj = series("projection", cn, grp, t)
            for s in seeds:
                ax[1].plot(pids, [pj.get(pid, {}).get(s) for pid in pids], marker="o", label=f"seed {s}")
            ax[1].set_title(f"Semantic Axis Projection Over Time (+ = {c.get('pole_a', ['Pole A'])[0]} side)", fontweight="bold")
            ax[1].legend(fontsize=8); ax[1].grid(True, alpha=0.3)
            
            # Panel 3: Target Word Frequency Confound Guardrail
            fq = series("frequency", cn, grp, t)
            for s in seeds[:1]:
                ax[2].plot(pids, [fq.get(pid, {}).get(s) for pid in pids], marker="o", color="black", label="Word Count")
            ax[2].axhline(CLAIM_FLOOR, color="red", linestyle="--", label=f"Claim Floor ({CLAIM_FLOOR})")
            ax[2].set_title("Word Frequency (Confound Guardrail)", fontweight="bold")
            ax[2].set_yscale("log"); ax[2].legend(fontsize=8); ax[2].grid(True, alpha=0.3)
            
            plt.xticks(rotation=30)
            fig.tight_layout()
            pp = OUTD / f"traj__{cn}__{str(grp).lower()}__{t}.png"
            fig.savefig(pp, dpi=120)
            plt.close(fig)
            made_plots.append(str(pp))

print(f"Generated {len(made_plots)} trajectory diagnostic plots in {OUTD}")

In [ ]:
# Cell 7 — Report Generator & RUN_SUMMARY.md Rollup
for c in CONCEPTS:
    cn = c["name"]
    L = [
        f"# Concept Report: {cn} ({today}, cfg-{cfg['config_version']})", "",
        "**Descriptive Diachronic Trajectory Analysis** (Stability bands across random seeds; no p-hacking).", ""
    ]
    for grp in sorted({g for (g, p) in order}):
        pids = sorted({p for (g, p) in order if g == grp})
        L.append(f"## Corpus Group: {grp} ({len(pids)} periods)")
        for pid in pids:
            cov = {s: cov_summary.get((cn, grp, pid, s), {}) for s in [1047, 2048, 9182] if (cn, grp, pid, s) in cov_summary}
            if len(cov) < 2:
                L.append(f"- **{pid}**: *Warning* — Single-seed cell; provisional confidence band.")
            for pole in ("pole_a", "pole_b"):
                miss = set()
                for s, cv in cov.items(): miss.update(cv.get(pole, {}).get("missing_or_rare", []))
                if miss:
                    L.append(f"- **{pid}**: {pole} missing/rare: {', '.join(sorted(miss))} (excluded from axis).")
        for t in c.get("targets", []):
            fq = series("frequency", cn, grp, t)
            lo = [p for p in pids for s, v in fq.get(p, {}).items() if v < CLAIM_FLOOR]
            if lo:
                L.append(f"- Target '{t}' below claim floor in: {', '.join(set(lo))}.")
        L.append("")
    
    L.append("## Criteria for Claiming a Reliable Semantic Shift:")
    L.append("1. **Seed Consistency**: Trajectory moves in the same direction across >=2 independent random seeds.")
    L.append("2. **Vocabulary Coverage**: Pole anchor words meet minimum frequency thresholds across all compared periods.")
    L.append("3. **Non-Frequency Confound**: Target word frequency remains stable (Panel 3) during the shift.")
    
    atomic_write_text(OUTD / f"report__{cn}.md", "\n".join(L) + "\n")
    print(f"Saved Concept Report -> diagnostics/semantic_axes/report__{cn}.md")

n_models = sum(1 for r in rows if r["status"] == "complete")
n_vec = sum(1 for r in rows if r["status"] == "complete" and r.get("vectors_path"))

summary = [
    f"# PROJECT RUN SUMMARY ({today}, config v{cfg['config_version']})", "",
    f"- **Completed Word2Vec Models**: {n_models}",
    f"- **Normalized Vector Exports (.kv)**: {n_vec}",
    f"- **Evaluated Concepts**: {len(CONCEPTS)}",
    f"- **Generated Trajectory Rows**: {len(trows_out)}",
    f"- **Generated Diagnostic Plots**: {len(made_plots)}",
    f"- **Master Manifest Registry**: `manifests/training_manifest.csv`",
    f"- **Reproducibility Sidecars**: `models/word2vec/**/*.nfo.json`", "",
    "All pipeline outputs are fully reproducible and idempotently tracked."
]
atomic_write_text(ROOT / "RUN_SUMMARY.md", "\n".join(summary) + "\n")
print("\n" + "=" * 70)
print("PROJECT RUN SUMMARY GENERATED (RUN_SUMMARY.md):")
print("=" * 70)
print("\n".join(summary))
print("=" * 70)